# Lab 10 — NER pipeline + hybrid rules

Practical NER pipeline on Ukrainian text with a Stanza baseline, a small manual evaluation set, and a hybrid rule layer for precision / coverage gains.

## 1. Install deps

In [1]:
from pathlib import Path
import subprocess
import sys

if Path('/content').exists() and not Path('/content/nlp_labs').exists():
    subprocess.run(['git', 'clone', 'https://github.com/velotsuraptor/nlp_labs.git', '/content/nlp_labs'], check=True)

project_root_candidates = [
    Path('/content/nlp_labs/project_lab10'),
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
]
PROJECT_ROOT = None
for cand in project_root_candidates:
    if (cand / 'requirements.txt').exists() and (cand / 'src').exists():
        PROJECT_ROOT = cand
        break
    if (cand / 'project_lab10' / 'requirements.txt').exists():
        PROJECT_ROOT = cand / 'project_lab10'
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate project_lab10/requirements.txt')

REPO_ROOT = PROJECT_ROOT.parent
req_path = PROJECT_ROOT / 'requirements.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req_path)], check=True)
print('project_root =', PROJECT_ROOT)

project_root = C:\Users\maia1\data\politiekh\masters\nlp\project_lab10


## 2. Data access

In [2]:
from pathlib import Path
import sys
import pandas as pd

ROOT = PROJECT_ROOT
sys.path.insert(0, str(REPO_ROOT))

from project_lab10.src.ner_pipeline import ensure_stanza_pipeline, load_eval_set, baseline_inference
from project_lab10.src.ner_rules import hybrid_entities, PHRASE_RULES
from project_lab10.src.ner_eval import (
    aggregate_entity_counts,
    build_error_frame,
    build_output_examples,
    compare_entities,
    entity_from_expected,
)

eval_rows = load_eval_set(ROOT / 'data' / 'sample' / 'eval_set_lab10.jsonl')
pd.DataFrame(eval_rows)[['text_id', 'label']].head()

,text_id,label
0,10001201,Question / Request for Help
1,3228,Gratitude / Positive Feedback
2,10000628,Suggestion / Idea
3,10002040,Suggestion / Idea
4,10003613,Question / Request for Help


## 3. Evaluation set preparation

In [3]:
print('evaluation_docs =', len(eval_rows))
print('pipeline_choice = Stanza Ukrainian NER')
eval_preview = pd.DataFrame([
    {
        'text_id': row['text_id'],
        'label': row['label'],
        'expected_entities': [(ent['text'], ent['label']) for ent in row['expected_entities']],
    }
    for row in eval_rows
])
eval_preview.head(10)

evaluation_docs = 20
pipeline_choice = Stanza Ukrainian NER


,text_id,label,expected_entities
0,10001201,Question / Request for Help,"[(18.04.2023, DATE)]"
1,3228,Gratitude / Positive Feedback,"[(15.11.2021, DATE)]"
2,10000628,Suggestion / Idea,"[(єВідновлення, DOMAIN)]"
3,10002040,Suggestion / Idea,"[(Наталіє, PERSON), (Дія, DOMAIN)]"
4,10003613,Question / Request for Help,"[(7 березня, DATE), (єВідновлення, DOMAIN), (8..."
5,10003153,Question / Request for Help,"[(Маріуполі, LOC)]"
6,10002513,Question / Request for Help,"[(єВідновлення, DOMAIN), (500 000, MONEY), (20..."
7,10001228,Question / Request for Help,"[(Повідомлення про пошкоджене майно, DOMAIN), ..."
8,7074,Suggestion / Idea,"[(400 грн, MONEY)]"
9,15893,Suggestion / Idea,"[(50 грн, MONEY), (60-80, MONEY), (КНТЕУ, ORG)]"


## 4. Load spaCy or Stanza pipeline

In [4]:
nlp = ensure_stanza_pipeline()
print('Chosen pipeline: Stanza uk tokenize+ner')
print('Out-of-the-box labels used in practice: PERS, ORG, LOC, MISC')

Chosen pipeline: Stanza uk tokenize+ner
Out-of-the-box labels used in practice: PERS, ORG, LOC, MISC


## 5. Baseline NER inference

In [5]:
records = []
for row in eval_rows:
    expected = [entity_from_expected(x) for x in row['expected_entities']]
    baseline_pred = baseline_inference(row['text'], nlp)
    records.append({
        'text_id': row['text_id'],
        'text': row['text'],
        'label': row['label'],
        'expected': expected,
        'baseline': {
            'predicted': baseline_pred,
            **compare_entities(expected, baseline_pred),
        },
    })

baseline_counts = aggregate_entity_counts(records, 'baseline')
baseline_counts

,label,correct,missed,false_positive,rough_precision,rough_recall
0,DATE,0,7,0,0.0,0.0000
1,DOMAIN,0,14,0,0.0,0.0000
2,LOC,1,1,1,0.5,0.5000
3,MISC,0,0,1,0.0,0.0000
4,MONEY,0,12,0,0.0,0.0000
5,ORG,1,2,1,0.5,0.3333
6,PERSON,1,1,0,1.0,0.5000


## 6. Inspect outputs

In [6]:
baseline_examples = build_output_examples(records, 'baseline')
baseline_examples.head(12)

,text_id,text,predicted_entities,expected_entities
0,10001201,Цифрова держава Повідомлення 123 від 18.04.202...,"[(Повідомлення 123, MISC), (Україні, LOC)]","[(18.04.2023, DATE)]"
1,3228,"Був у центрі 15.11.2021року Просто супер, жінк...",[],"[(15.11.2021, DATE)]"
2,10000628,"Доброго дня, спробуйте при створенні замовленн...",[],"[(єВідновлення, DOMAIN)]"
3,10002040,"Наталіє, якщо у людини немає можливості автори...",[],"[(Наталіє, PERSON), (Дія, DOMAIN)]"
4,10003613,Добрий день! 7 березня послуга єВідновлення не...,[],"[(7 березня, DATE), (єВідновлення, DOMAIN), (8..."
5,10003153,А коли почне діяти програма компенсацій за зру...,"[(Маріуполі, LOC)]","[(Маріуполі, LOC)]"
6,10002513,Який розмір грошової компенсації за пошкоджене...,[],"[(єВідновлення, DOMAIN), (500 000, MONEY), (20..."
7,10001228,"Зверніть увагу, подати ""Повідомлення про пошко...","[(Повідомлення про пошкоджене майно, MISC), (Ц...","[(Повідомлення про пошкоджене майно, DOMAIN), ..."
8,7074,Будьте уважні! При оформленні паспорта вартіст...,[],"[(400 грн, MONEY)]"
9,15893,Вхід 50 грн або екскурсія + прогулянка 60-80 М...,"[(КНТЕУ 😆😅😅…, ORG)]","[(50 грн, MONEY), (60-80, MONEY), (КНТЕУ, ORG)]"


## 7. Add hybrid rules

In [7]:
rule_summary = pd.DataFrame([
    {'rule_type': 'regex', 'label': 'DATE', 'description': 'Numeric and textual date extraction'},
    {'rule_type': 'regex', 'label': 'MONEY', 'description': 'Money patterns such as 400 грн, 80грн, 500 000'},
    {'rule_type': 'phrase dictionary', 'label': 'DOMAIN / ORG / LOC', 'description': 'Corpus-specific entities: єВідновлення, Дія, ЦНАП, КНТЕУ, Ратуша, etc.'},
])
rule_summary

,rule_type,label,description
0,regex,DATE,Numeric and textual date extraction
1,regex,MONEY,"Money patterns such as 400 грн, 80грн, 500 000"
2,phrase dictionary,DOMAIN / ORG / LOC,"Corpus-specific entities: єВідновлення, Дія, Ц..."


## 8. Run hybrid inference

In [8]:
for record in records:
    hybrid_pred = hybrid_entities(record['text'], record['baseline']['predicted'])
    record['hybrid'] = {
        'predicted': hybrid_pred,
        **compare_entities(record['expected'], hybrid_pred),
    }

hybrid_counts = aggregate_entity_counts(records, 'hybrid')
hybrid_counts

,label,correct,missed,false_positive,rough_precision,rough_recall
0,DATE,7,0,0,1.0000,1.0
1,DOMAIN,14,0,1,0.9333,1.0
2,LOC,2,0,1,0.6667,1.0
3,MISC,0,0,1,0.0000,0.0
4,MONEY,12,0,1,0.9231,1.0
5,ORG,3,0,1,0.7500,1.0
6,PERSON,1,1,0,1.0000,0.5


## 9. Compare baseline vs hybrid

In [9]:
comparison_df = baseline_counts.merge(hybrid_counts, on='label', how='outer', suffixes=('_baseline', '_hybrid')).fillna(0)
comparison_df

,label,correct_baseline,missed_baseline,false_positive_baseline,rough_precision_baseline,rough_recall_baseline,correct_hybrid,missed_hybrid,false_positive_hybrid,rough_precision_hybrid,rough_recall_hybrid
0,DATE,0,7,0,0.0,0.0000,7,0,0,1.0000,1.0
1,DOMAIN,0,14,0,0.0,0.0000,14,0,1,0.9333,1.0
2,LOC,1,1,1,0.5,0.5000,2,0,1,0.6667,1.0
3,MISC,0,0,1,0.0,0.0000,0,0,1,0.0000,0.0
4,MONEY,0,12,0,0.0,0.0000,12,0,1,0.9231,1.0
5,ORG,1,2,1,0.5,0.3333,3,0,1,0.7500,1.0
6,PERSON,1,1,0,1.0,0.5000,1,1,0,1.0000,0.5


## 10. Error analysis

In [10]:
baseline_errors = build_error_frame(records, 'baseline')
hybrid_errors = build_error_frame(records, 'hybrid')
baseline_errors['system'] = 'baseline'
hybrid_errors['system'] = 'hybrid'
error_analysis_df = pd.concat([
    baseline_errors.head(9),
    hybrid_errors.head(6),
], ignore_index=True)

print('baseline_error_count =', len(baseline_errors))
print('hybrid_error_count =', len(hybrid_errors))
print('combined_displayed_errors =', len(error_analysis_df))
error_category_summary = pd.concat([
    baseline_errors.assign(system='baseline'),
    hybrid_errors.assign(system='hybrid')
]).groupby(['system', 'category']).size().reset_index(name='count').sort_values(['system', 'count'], ascending=[True, False])

display(error_category_summary)
error_analysis_df

baseline_error_count = 40
hybrid_error_count = 6
combined_displayed_errors = 15


,system,category,count
3,baseline,missed entity,22
2,baseline,missed domain entity,12
1,baseline,false positive,3
4,baseline,type error,2
0,baseline,boundary error,1
5,hybrid,false positive,5
6,hybrid,missed entity,1


,text_id,text_excerpt,expected_entity,expected_type,predicted_entity,predicted_type,category,explanation,system
0,10001201,Цифрова держава Повідомлення 123 від 18.04.202...,18.04.2023,DATE,,,missed entity,Expected entity was not extracted at all.,baseline
1,10001201,Цифрова держава Повідомлення 123 від 18.04.202...,,,Повідомлення 123,MISC,false positive,The system predicted an entity where no gold e...,baseline
2,10001201,Цифрова держава Повідомлення 123 від 18.04.202...,,,Україні,LOC,false positive,The system predicted an entity where no gold e...,baseline
3,3228,"Був у центрі 15.11.2021року Просто супер, жінк...",15.11.2021,DATE,,,missed entity,Expected entity was not extracted at all.,baseline
4,10000628,"Доброго дня, спробуйте при створенні замовленн...",єВідновлення,DOMAIN,,,missed domain entity,Baseline or hybrid missed a corpus-specific en...,baseline
5,10002040,"Наталіє, якщо у людини немає можливості автори...",Наталіє,PERSON,,,missed entity,Expected entity was not extracted at all.,baseline
6,10002040,"Наталіє, якщо у людини немає можливості автори...",Дія,DOMAIN,,,missed domain entity,Baseline or hybrid missed a corpus-specific en...,baseline
7,10003613,Добрий день! 7 березня послуга єВідновлення не...,7 березня,DATE,,,missed entity,Expected entity was not extracted at all.,baseline
8,10003613,Добрий день! 7 березня послуга єВідновлення не...,єВідновлення,DOMAIN,,,missed domain entity,Baseline or hybrid missed a corpus-specific en...,baseline
9,10001201,Цифрова держава Повідомлення 123 від 18.04.202...,,,Повідомлення 123,MISC,false positive,The system predicted an entity where no gold e...,hybrid


## 11. Generate docs/audit_summary_lab10.md

In [11]:
from pathlib import Path


docs_dir = ROOT / 'docs'
docs_dir.mkdir(parents=True, exist_ok=True)
(ROOT / 'labs' / 'lab10').mkdir(parents=True, exist_ok=True)

baseline_total_correct = int(baseline_counts['correct'].sum())
baseline_total_missed = int(baseline_counts['missed'].sum())
baseline_total_fp = int(baseline_counts['false_positive'].sum())
hybrid_total_correct = int(hybrid_counts['correct'].sum())
hybrid_total_missed = int(hybrid_counts['missed'].sum())
hybrid_total_fp = int(hybrid_counts['false_positive'].sum())

summary_md = f'''# Audit summary — Lab10

1. Pipeline used: Stanza Ukrainian NER (`tokenize,ner`).
2. Important entity types for this corpus: PERSON, ORG, LOC, DATE, MONEY, and domain entities such as `єВідновлення` / `Дія`.
3. Baseline strengths: it already catches some PERSON / ORG / LOC entities.
4. Baseline misses: almost all DATE, MONEY, and corpus-specific domain entities.
5. Added rules: date regex, money regex, and phrase-dictionary rules for domain entities / ORG / LOC.
6. What improved: baseline correct={baseline_total_correct}, missed={baseline_total_missed}, false_positive={baseline_total_fp}; hybrid correct={hybrid_total_correct}, missed={hybrid_total_missed}, false_positive={hybrid_total_fp}.
7. Most frequent error categories: baseline mainly missed entity / missed domain entity; hybrid mainly false positives plus one remaining PERSON miss.
8. Next steps: better PERSON coverage for vocative forms, tighter filtering for hotline-like numbers, and stronger conflict resolution for baseline MISC spans.
'''
(docs_dir / 'audit_summary_lab10.md').write_text(summary_md, encoding='utf-8')

notes_lines = []
notes_lines.append('# NER notes — Lab10')
notes_lines.append('')
notes_lines.append('## 1. Chosen pipeline')
notes_lines.append('')
notes_lines.append('- Stanza Ukrainian NER pipeline (`tokenize,ner`)')
notes_lines.append('- Out-of-the-box labels observed: PERS, ORG, LOC, MISC')
notes_lines.append('')
notes_lines.append('## 2. Important entities in this corpus')
notes_lines.append('')
notes_lines.append('- PERSON: named lawyers and users')
notes_lines.append('- ORG / LOC: ЦНАП, КНТЕУ, Ратуша, Маріуполі')
notes_lines.append('- DATE / MONEY: dates, fees, compensation amounts')
notes_lines.append('- DOMAIN: єВідновлення, Дія, Повідомлення про пошкоджене майно')
notes_lines.append('')
notes_lines.append('## 3. Added rules')
notes_lines.append('')
notes_lines.append('- Regex for DATE entities')
notes_lines.append('- Regex for MONEY entities')
notes_lines.append('- Phrase dictionary for domain entities and corpus-specific ORG / LOC names')
notes_lines.append('')
notes_lines.append('## 4. What improved after rules')
notes_lines.append('')
for _, row in comparison_df.iterrows():
    notes_lines.append(
        f'- {row["label"]}: baseline correct={int(row["correct_baseline"])}, missed={int(row["missed_baseline"])}, fp={int(row["false_positive_baseline"])}, hybrid correct={int(row["correct_hybrid"])}, missed={int(row["missed_hybrid"])}, fp={int(row["false_positive_hybrid"])}'
    )
notes_lines.append('')
notes_lines.append('## 5. Remaining errors')
notes_lines.append('')
for _, row in hybrid_errors.iterrows():
    notes_lines.append(f'- text_id={int(row["text_id"])} | {row["category"]} | expected={row["expected_entity"]} | predicted={row["predicted_entity"]}')
notes_lines.append('')
notes_lines.append('## 6. What to fix next')
notes_lines.append('')
notes_lines.append('- Add a small vocative-name lexicon or person-name fallback rules')
notes_lines.append('- Tighten MONEY rules around hotline / service number patterns')
notes_lines.append('- Add postprocessing to suppress baseline MISC spans like `Повідомлення 123`')
(docs_dir / 'ner_notes_lab10.md').write_text('\n'.join(notes_lines), encoding='utf-8')

dataset_card_md = f'''# Dataset card — Lab10

## NER relevance
- Important entity types in this corpus: PERSON, ORG, LOC, DATE, MONEY, and domain entities such as `єВідновлення` and `Дія`.
- Standard Ukrainian NER baseline is not sufficient on its own: it catches some classic entities, but misses many domain and regular-pattern entities.
- The most problematic entities were domain-specific names plus DATE and MONEY mentions in noisy user text.
- Hybrid rules gave a clear gain in coverage and practical usefulness, especially for DATE, MONEY, and corpus-specific domain entities.
- Remaining issues: vocative person names, hotline-like numbers, and some baseline false positives from MISC / LOC spans.
'''
(docs_dir / 'dataset_card.md').write_text(dataset_card_md, encoding='utf-8')

readme_md = f'''# LPNU NLP — Lab 10 (NER pipeline + hybrid rules)

1. Evaluation set: 20 short documents from the cleaned `processed_v2` corpus with manual expected entities.
2. Pipeline: Stanza Ukrainian NER (`tokenize,ner`).
3. Added rules: DATE regex, MONEY regex, phrase dictionary for domain entities / ORG / LOC.
4. Baseline strengths: some PERSON / ORG / LOC detection.
5. Baseline misses: most DATE, MONEY, and corpus-specific domain entities.
6. Hybrid gains: large recall improvement for DATE, MONEY, DOMAIN, and better ORG / LOC coverage.
7. Remaining issues: false positives from baseline spans, one missed PERSON case, and ambiguous hotline-like number patterns.
'''
(ROOT / 'labs' / 'lab10' / 'README.md').write_text(readme_md, encoding='utf-8')

print((docs_dir / 'audit_summary_lab10.md').read_text(encoding='utf-8'))

# Audit summary — Lab10

1. Pipeline used: Stanza Ukrainian NER (`tokenize,ner`).
2. Important entity types for this corpus: PERSON, ORG, LOC, DATE, MONEY, and domain entities such as `єВідновлення` / `Дія`.
3. Baseline strengths: it already catches some PERSON / ORG / LOC entities.
4. Baseline misses: almost all DATE, MONEY, and corpus-specific domain entities.
5. Added rules: date regex, money regex, and phrase-dictionary rules for domain entities / ORG / LOC.
6. What improved: baseline correct=3, missed=37, false_positive=3; hybrid correct=39, missed=1, false_positive=5.
7. Most frequent error categories: baseline mainly missed entity / missed domain entity; hybrid mainly false positives plus one remaining PERSON miss.
8. Next steps: better PERSON coverage for vocative forms, tighter filtering for hotline-like numbers, and stronger conflict resolution for baseline MISC spans.

